# Build Your Own Tiny LLM from Scratch

Welcome! In this hands-on tutorial, we are going to build, understand, and train a complete decoder-only **Transformer-based Language Model** from scratch using plain PyTorch.

There are no external frameworks (like HuggingFace transformers, peft, or trl) hidden under the hood. Everything is written explicitly so that you can see exactly how characters are turned into tokens, how multi-head self-attention works mathematically, how the transformer block is constructed, and how we train it on a CPU on a normal laptop.

## Table of Contents
1. Introduction
2. What is a language model?
3. Dataset
4. Tokenization
5. Token IDs
6. Embeddings
7. Positional information
8. Self-attention
9. Multi-head attention
10. Feed-forward network
11. Transformer block
12. Language-model head
13. Cross-entropy loss
14. Backpropagation
15. Training loop
16. Checkpoints
17. Text generation
18. Experiment with hyperparameters
19. Save the trained model

## 1. Introduction

The modern AI revolution is powered by the **Transformer** architecture. While commercial models contain billions or trillions of parameters and require massive GPU clusters, the mathematical principles remain identical. By building and training a tiny (~100K parameters) model on a small dataset like Tiny Shakespeare, you can learn how these components work in real time, right on your CPU.

## 2. What is a language model?

At its core, a autoregressive (decoder-only) language model does one thing: **predicts the next token given a sequence of tokens**.

$$\text{Prompt: } \text{"To be, or not to "} \longrightarrow \text{Model} \longrightarrow \text{Prediction: } \text{"be"}$$

By generating one word (or token) at a time, appending it to the prompt, and repeating the process, the model generates complete passages of text.

## 3. Dataset

We will use a slice of **Tiny Shakespeare** as our dataset. Let's write a small script to download a tiny, manageable piece of the Shakespeare dataset if we don't have it locally.

In [ ]:
import os
import urllib.request

# Create data directory if it doesn't exist
os.makedirs("data/student", exist_ok=True)
txt_path = "data/student/train.txt"

if os.path.exists(txt_path):
    print(f"Dataset already available locally at {txt_path}!")
else:
    print("Downloading Tiny Shakespeare dataset...")
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, txt_path)
    print("Download complete!")

# Read and display the first 250 characters of the text
with open(txt_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f"Total characters in dataset: {len(raw_text):,}")
print("\n--- First 250 Characters of raw text ---")
print(raw_text[:250])

## 4. Tokenization

Computers cannot process raw characters directly; they process numbers. A **Tokenizer** is responsible for converting text strings into a list of integers, and vice-versa.

We will use `tiktoken` (specifically the `r50k_base` vocabulary used by OpenAI GPT-3) to encode our text into token IDs.

In [ ]:
import tiktoken

# Load encoding
enc = tiktoken.get_encoding("r50k_base")

sample_phrase = "To be, or not to be?"
encoded_sample = enc.encode(sample_phrase)
decoded_sample = enc.decode(encoded_sample)

print(f"Original text:   '{sample_phrase}'")
print(f"Token IDs:       {encoded_sample}")
print(f"Decoded text:     '{decoded_sample}'")
print("\nIndividual tokens matching IDs:")
for tid in encoded_sample:
    print(f"  {tid:5d} -> '{enc.decode([tid])}'")

## 5. Token IDs

Let's tokenize a larger portion of the dataset and organize it into PyTorch Tensors. For educational clarity, we'll extract a subset of 50,000 tokens.

In [ ]:
import torch

# Tokenize a slice of the text
tokens = enc.encode(raw_text[:200000])
data_tensor = torch.tensor(tokens, dtype=torch.long)

print(f"Subset dataset token count: {len(data_tensor):,}")
print("First 15 token IDs:", data_tensor[:15])

## 6. Embeddings

Each Token ID must be mapped to a continuous vector of numbers (the embedding space). A simple lookup table `nn.Embedding` handles this, converting discrete token IDs into dense vectors of dimension `n_embed`.

In [ ]:
import torch.nn as nn

vocab_size = 50304  # Standard vocab size
n_embed = 32        # Tiny embedding size for our illustration

token_embedding_table = nn.Embedding(vocab_size, n_embed)

# Let's embed a batch of size 2 with context length 4
dummy_input = torch.randint(0, vocab_size, (2, 4))
embedded = token_embedding_table(dummy_input)

print("Input shape:    ", dummy_input.shape) # (Batch Size, Context Length)
print("Embedded shape: ", embedded.shape)    # (Batch Size, Context Length, Embedding Size)

## 7. Positional information

Since Transformers process all tokens in parallel, they have no native notion of word order. We must inject position vectors to help the model learn where each token sits relative to other tokens.

In [ ]:
context_length = 8
position_embedding_table = nn.Embedding(context_length, n_embed)

# Create positions [0, 1, 2, ..., context_length-1]
positions = torch.arange(context_length)
pos_emb = position_embedding_table(positions)

print("Positions vector:", positions)
print("Position Embeddings shape:", pos_emb.shape)

## 8. Self-attention

Self-attention allows tokens to dynamically look at other tokens to build context.

For each token, we generate:
- **Query (Q)**: What information am I seeking?
- **Key (K)**: What information do I offer?
- **Value (V)**: If selected, what information do I communicate?

Attention weights are calculated using the query-key dot product, scaled, masked to prevent looking into the future (causal masking), and used to sum up the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V$$

In [ ]:
import torch.nn.functional as F
import math

class SingleAttentionHead(nn.Module):
    def __init__(self, head_size, n_embed, context_length):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        # lower-triangular matrix used for causal masking
        self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # Score similarities
        scores = q @ k.transpose(-2, -1) / math.sqrt(C) # (B, T, T)
        
        # Apply causal masking so token t cannot attend to t+1, t+2...
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        
        # Convert scores to weights
        weights = F.softmax(scores, dim=-1) # (B, T, T)
        
        v = self.value(x) # (B, T, head_size)
        out = weights @ v # (B, T, head_size)
        return out

# Test Single Head
head = SingleAttentionHead(head_size=16, n_embed=32, context_length=8)
dummy_x = torch.randn(2, 8, 32)
out = head(dummy_x)
print("Single Head Attention Output Shape:", out.shape)

## 9. Multi-head attention

We can run multiple attention heads in parallel. This allows the model to capture multiple types of relationships simultaneously (e.g. subject-verb agreement and pronoun references).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embed, context_length):
        super().__init__()
        self.heads = nn.ModuleList([
            SingleAttentionHead(head_size, n_embed, context_length) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(num_heads * head_size, n_embed)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

mha = MultiHeadAttention(num_heads=4, head_size=8, n_embed=32, context_length=8)
out_mha = mha(dummy_x)
print("Multi-Head Attention Output Shape:", out_mha.shape)

## 10. Feed-forward network

After aggregating context across positions using multi-head attention, we apply a non-linear fully-connected projection (Multi-Layer Perceptron) individually to each token. This allows the model to compute complex representations.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed)
        )

    def forward(self, x):
        return self.net(x)

ffn = FeedForward(n_embed=32)
out_ffn = ffn(dummy_x)
print("FeedForward Output Shape:", out_ffn.shape)

## 11. Transformer block

The complete **Transformer Block** stacks Multi-Head Attention and the Feed-Forward network, surrounded by Pre-LayerNormalization and residual connections.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, n_embed, n_head, context_length):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embed, context_length)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        # Attention with Pre-LN & Residual connection
        x = x + self.sa(self.ln1(x))
        # Feed-forward with Pre-LN & Residual connection
        x = x + self.ffwd(self.ln2(x))
        return x

## 12. Language-model head

To map our hidden states back into vocabulary predictions, we stack multiple Transformer blocks, normalize the final output, and use a linear projection (`lm_head`) to map embedding states of size `n_embed` to token probabilities of size `vocab_size`.

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_blocks, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embed = nn.Embedding(vocab_size, n_embed)
        self.position_embed = nn.Embedding(context_length, n_embed)
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embed, n_head, context_length) for _ in range(n_blocks)
        ])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.token_embed(idx) # (B, T, n_embed)
        pos_emb = self.position_embed(torch.arange(T, device=idx.device)) # (T, n_embed)
        x = tok_emb + pos_emb # (B, T, n_embed)
        
        x = self.blocks(x) # (B, T, n_embed)
        x = self.ln_f(x)   # (B, T, n_embed)
        logits = self.lm_head(x) # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
            
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 13. Cross-entropy loss

The cross-entropy loss measures how "surprised" the model was by the actual next-token target relative to its logits distribution. Minimizing this loss guides the model to predict the training text correctly.

## 14. Backpropagation

During backpropagation, PyTorch calculates the gradients of the cross-entropy loss with respect to all the model's parameters (weights and biases), adjusting them slightly via an optimizer (AdamW) to improve next-token predictions.

## 15. Training loop

Let's write a simple training loop to train our `TinyTransformerLM` on our dataset using CPU. We'll set small parameters to ensure it finishes quickly.

In [ ]:
import random

# Define small dimensions for CPU feasibility
vocab_size = 50304
n_embed = 64
n_head = 2
n_blocks = 2
context_length = 32
batch_size = 4

device = "cpu"
model = TinyTransformerLM(vocab_size, n_embed, n_head, n_blocks, context_length).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Simple dataloader
def get_batch(data, batch_size, context_length):
    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i+context_length] for i in ix])
    y = torch.stack([data[i+1:i+context_length+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Training start. Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

for step in range(100):
    xb, yb = get_batch(data_tensor, batch_size, context_length)
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if (step + 1) % 10 == 0 or step == 0:
        print(f"Step {step+1:3d}/100 | Train Loss: {loss.item():.4f}")

## 16. Checkpoints

Saving checkpoints lets us resume training later and preserve progress. We can save a state dictionary easily with PyTorch.

In [ ]:
# Save training checkpoint metadata
checkpoint_data = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'step': 100
}
os.makedirs("checkpoints", exist_ok=True)
torch.save(checkpoint_data, "checkpoints/byo_tiny_llm.pt")
print("Checkpoint saved successfully!")

## 17. Text generation

Now let's see how our trained model generates text starting from a simple prompt!

In [ ]:
prompt = "GLOUCESTER:"
input_ids = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long, device=device)

generated_ids = model.generate(input_ids, max_new_tokens=25)[0].tolist()
generated_text = enc.decode(generated_ids)

print("--- Generated Output ---")
print(generated_text)

## 18. Experiment with hyperparameters

In the project root, you can find preset configurations for:
- `tiny`: ~100K-300K params
- `student`: ~1.5M params
- `small`: ~5.8M params
- `gpu`: ~77M+ params

Try tweaking `n_embed`, `n_head`, and `n_blocks` in your own scripts to observe their impact on performance and learning speed.

## 19. Save the trained model

We have already saved a final checkpoint. You can load this checkpoint back at any time and resume writing Shakespearean-style text!

Congratulations, you have fully built, analyzed, and trained a custom-made Transformer Language Model entirely on your CPU!